In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross

In [17]:
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')

In [18]:
le=LabelEncoder()

X=df_train.drop(columns=['health_condition'])
y = le.fit_transform(df_train['health_condition']) 
test_ids = df_test['id'].copy()

In [19]:
def features(df):
    df=df.copy()
    df.drop(columns='id')
    df['stress_level_NaN']=df_train['stress_level'].isna().astype(bool)
    df['sleep_duration']=df['sleep_duration'].isna().astype(bool)
    df['physical_activity_level']=df['physical_activity_level'].isna().astype(bool)
    df['smoking_alcohol']=df['smoking_alcohol'].isna().astype(bool)
    df['calorie_expenditure']=df['calorie_expenditure'].isna().astype(bool)
    df['water_intake']=df['water_intake'].isna().astype(bool)
    return df

features_eng=FunctionTransformer(features)

In [20]:
ord_cols=['sleep_quality','stress_level','smoking_alcohol','physical_activity_level']

obj_cols=X.select_dtypes(include=['object']).columns
cat_cols=[c for c in obj_cols if c not in ord_cols]

bool_cols=X.select_dtypes(include=['bool']).columns
num_cols=X.select_dtypes(include=['number']).columns

In [21]:
print(ord_cols)
print(cat_cols)
print(bool_cols)
print(num_cols)

['sleep_quality', 'stress_level', 'smoking_alcohol', 'physical_activity_level']
['diet_type', 'gender']
Index([], dtype='object')
Index(['id', 'sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
       'step_count', 'exercise_duration', 'water_intake'],
      dtype='object')


In [22]:
ordi_scales={
    'sleep_quality':['average', 'poor', 'Inconnu', 'good'],
    'stress_level':['high', 'low', 'Inconnu', 'medium'],
    'smoking_alcohol':['yes', 'occasional', 'Inconnu', 'no'],
    'physical_activity_level':['sedentary', 'moderate', 'active', 'Inconnu']
}

In [ ]:
ordinal_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
('OrdinalEncoder',OrdinalEncoder(categories=[ordi_scales[c] for c in ord_cols])),
('scaler',StandardScaler())
])

cat_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
('OneHotEncoder',OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
('scaler',StandardScaler())
])

num_pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),
('scaler',StandardScaler())
])

In [ ]:
preprocessor=ColumnTransformer([('ordinal_pipe',ordinal_pipe,ord_cols),('cat_pipe',cat_pipe,cat_cols),('num_pipe',num_pipe,num_cols)])

In [ ]:
logistic_pipe=Pipeline([('features_eng',features_eng),('preprocessor',preprocessor),('logistic',LogisticRegression(random_state=42,max_iter=1000))])